In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

# ==========================================
# 1. 트랜스포머 인코더 블록 정의 (레고 블록)
# ==========================================
def transformer_encoder_block(embed_dim, num_heads, ff_dim, rate=0.1):
    inputs = layers.Input(shape=(None, embed_dim))
    
    # [Step 1] Multi-Head Attention (단어 간 관계 파악)
    # 전문가(Head)들이 각기 다른 관점으로 문장을 분석합니다.
    attention_output = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_dim
    )(inputs, inputs)
    attention_output = layers.Dropout(rate)(attention_output)
    
    # Residual Connection + Layer Norm (정보 보존 및 정규화)
    out1 = layers.LayerNormalization(epsilon=1e-6)(inputs + attention_output)
    
    # [Step 2] Feed Forward Network (데이터 의미 심화 학습)
    ffn_output = layers.Dense(ff_dim, activation="relu")(out1)
    ffn_output = layers.Dense(embed_dim)(ffn_output)
    ffn_output = layers.Dropout(rate)(ffn_output)
    
    # Residual Connection + Layer Norm
    return tf.keras.Model(inputs=inputs, outputs=layers.LayerNormalization(epsilon=1e-6)(out1 + ffn_output))

# ==========================================
# 2. 위치 정보(Position) 포함 임베딩 층
# ==========================================
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions
# ==========================================
# 3. 데이터 준비 (스마트 팩토리 가상 로그 - 한글 샘플)
# ==========================================
# 실제 강의 시에는 CSV 파일을 로드하거나 아래와 같이 샘플을 만듭니다.
vocab_size = 5000  # 단어 사전 크기
maxlen = 20        # 문장 최대 길이

# 실제로는 엑셀(CSV) 파일을 불러오지만, 여기서는 실습을 위해 직접 정의
# 적절한 학습이 되기 위해서는 Big Data 필요 수만~수백만 데이터 필요
# 아래 소량의 데이터로는 학습이 전혀 안됨
# [정상 로그: 라벨 0]
normal_logs = [
    "장비 가동 상태 정상", "현재 온도 25도 유지 중", "진동 수치 안정적임",
    "정기 점검 완료 이상 없음", "모터 회전수 정상 범위", "센서 데이터 송신 중",
    "시스템 전원 온료", "주변 습도 최적화 상태", "라인 가동율 95퍼센트",
    "냉각 시스템 정상 작동"
] 
# [위험/불량 로그: 라벨 1]
danger_logs = [
    "급격한 온도 상승 감지", "모터 소음 비정상 발생", "센서 연결 끊김 위험",
    "긴급 정지 버튼 작동", "진동 수치 임계치 초과", "부품 마모 심함 교체 필요",
    "오일 누유 발생 즉시 확인", "배선 합선 우려 전원 차단 필요", "생산품 불량률 급증",
    "장비 과부하 경고 발생"
] 
# 데이터 합치기
all_texts = normal_logs + danger_logs
all_labels = [0] * len(normal_logs) + [1] * len(danger_logs)

# 텍스트를 숫자로 바꾸는 '단어 사전' 만들기 (강의 핵심!)
vectorize_layer = layers.TextVectorization(
    max_tokens=2000,
    output_mode='int',
    output_sequence_length=maxlen
)
vectorize_layer.adapt(all_texts) # 단어 공부시키기

# 문장을 숫자로 변환
X_train = vectorize_layer(all_texts)
y_train = np.array(all_labels)

print(f"학습 데이터 준비 완료: 총 {len(all_texts)}개의 로그")
# ==========================================
# 4. 전체 모델 조립 (Classification 모델)
# ==========================================
embed_dim = 32  # 단어당 표현 숫자 개수
num_heads = 2   # 어텐션 헤드 개수
ff_dim = 32     # 내부 신경망 크기

inputs = layers.Input(shape=(maxlen,))
# 1단계: 단어 의미 + 위치 정보 부여
x = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)(inputs)
# 2단계: 트랜스포머 블록 통과 (핵심 연산)
x = transformer_encoder_block(embed_dim, num_heads, ff_dim)(x)
# 3단계: 문장 정보 압축 및 출력 (Global Average Pooling)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(20, activation="relu")(x)
outputs = layers.Dense(2, activation="softmax")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

# 모델 컴파일
model.compile(optimizer="adam", 
              loss="sparse_categorical_crossentropy", 
              metrics=["accuracy"])

# 모델 구조 출력
model.summary()

# ==========================================
# 5. 학습 및 액셔너블 AI 테스트 함수
# ==========================================
print("\n--- 모델 학습 시작 ---")
model.fit(X_train, y_train, epochs=20, batch_size=8)

def predict_and_action(text_list):
    # 실제 강의 시에는 여기에 TextVectorization 과정을 포함시킵니다.
    # 여기서는 구조 이해를 위해 가상 숫자로 테스트합니다.
    #sample_input = np.random.randint(1, vocab_size, (1, maxlen))
    #prediction = model.predict(sample_input)
    #danger_prob = prediction[0][1]

    # 1. 실제 텍스트를 모델이 이해할 수 있는 숫자(Vector)로 변환
    # vectorize_layer는 학습 데이터로 미리 .adapt() 되어 있어야 합니다.
    tokenized_input = vectorize_layer(text_list)
    
    # 2. 모델 예측 실행
    prediction = model.predict(tokenized_input, verbose=0)
    danger_prob = prediction[0][1]

    
    print(f"입력 : {text_list}")
    
    print(f"\n[입력 로그 분석 결과]")
    print(f"위험 감지 확률: {danger_prob*100:.2f}%")
    
    if danger_prob > 0.5:
        print("🚨 ACTION: 장비 가동 중단 및 정비팀 알림 전송!")
    else:
        print("✅ STATUS: 정상 가동 유지")

    print("\n\n\n")

# 테스트 실행
predict_and_action(["모터 진동 이상 감지"])

학습 데이터 준비 완료: 총 20개의 로그


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)           │ (None, 20)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ token_and_position_embedding_2       │ (None, 20, 32)              │         160,640 │
│ (TokenAndPositionEmbedding)          │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ functional_4 (Functional)            │ (None, 20, 32)              │          10,656 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d_2           │ (None, 32)                  │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_11 (Dropout)                 │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (None, 20)                  │             660 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_11 (Dense)                     │ (None, 2)                   │              42 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 171,998 (671.87 KB)

 Trainable params: 171,998 (671.87 KB)

 Non-trainable params: 0 (0.00 B)


--- 모델 학습 시작 ---
Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.4000 - loss: 0.7914
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5500 - loss: 0.7265
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5000 - loss: 0.7327
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6000 - loss: 0.7023
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5000 - loss: 0.6802
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6000 - loss: 0.6226
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6500 - loss: 0.6104
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7500 - loss: 0.5848
Epoch 9/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6000 - loss: 0.6102
Epoch 10/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.7500 - loss: 0.5637
Epoch 11/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.7000 - loss: 0.6102
Epoch 12/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.650

In [7]:
# 테스트 실행
# 안정
predict_and_action(["모터 진동 이상 감지되었지만 동작 이상 없는 수준"])
predict_and_action(["장비 안정적 유지중."])
predict_and_action(["장비 가동 상태 정상."])
# 위험
predict_and_action(["불량 증가. 센서 정보 확인 필요"])
predict_and_action(["불량 발샹. 즉시 센서 정보 확인 필요. 생산 불량 발생 가능"])
predict_and_action(["급격한 불량 증가. 즉시 센서 정보 확인 필요. 장비 계속 구동시 생산 불량 발생 가능"])


predict_and_action(["기분이 좋고 상쾌합니다"])
predict_and_action(["상황이 좋아서 크게 걱정 없습니다"])

predict_and_action(["마음이 불안하고 걱정됩니다"])
predict_and_action(["무슨일이 일어날지도 모릅니다"])

입력 : ['모터 진동 이상 감지되었지만 동작 이상 없는 수준']

[입력 로그 분석 결과]
위험 감지 확률: 40.15%
✅ STATUS: 정상 가동 유지




입력 : ['장비 안정적 유지중.']

[입력 로그 분석 결과]
위험 감지 확률: 39.98%
✅ STATUS: 정상 가동 유지




입력 : ['장비 가동 상태 정상.']

[입력 로그 분석 결과]
위험 감지 확률: 15.52%
✅ STATUS: 정상 가동 유지




입력 : ['불량 증가. 센서 정보 확인 필요']

[입력 로그 분석 결과]
위험 감지 확률: 68.20%
🚨 ACTION: 장비 가동 중단 및 정비팀 알림 전송!




입력 : ['불량 발샹. 즉시 센서 정보 확인 필요. 생산 불량 발생 가능']

[입력 로그 분석 결과]
위험 감지 확률: 88.26%
🚨 ACTION: 장비 가동 중단 및 정비팀 알림 전송!




입력 : ['급격한 불량 증가. 즉시 센서 정보 확인 필요. 장비 계속 구동시 생산 불량 발생 가능']

[입력 로그 분석 결과]
위험 감지 확률: 90.19%
🚨 ACTION: 장비 가동 중단 및 정비팀 알림 전송!




입력 : ['기분이 좋고 상쾌합니다']

[입력 로그 분석 결과]
위험 감지 확률: 42.27%
✅ STATUS: 정상 가동 유지




입력 : ['상황이 좋아서 크게 걱정 없습니다']

[입력 로그 분석 결과]
위험 감지 확률: 46.79%
✅ STATUS: 정상 가동 유지




입력 : ['마음이 불안하고 걱정됩니다']

[입력 로그 분석 결과]
위험 감지 확률: 42.27%
✅ STATUS: 정상 가동 유지




입력 : ['무슨일이 일어날지도 모릅니다']

[입력 로그 분석 결과]
위험 감지 확률: 42.27%
✅ STATUS: 정상 가동 유지




